# Assignment 1: SVM and Naive Bayes



## Theory Q and A

**1) What is a Support Vector Machine (SVM)**
An SVM is a supervised learning algorithm that finds a decision boundary (hyperplane) that maximizes the margin between classes. It can also be used for regression (SVR).

**2) Difference between Hard Margin and Soft Margin SVM**
Hard margin requires perfect separation (no misclassification) and only works when classes are linearly separable. Soft margin allows some misclassification with a penalty controlled by `C`, improving robustness to noise and overlap.

**3) Mathematical intuition behind SVM**
SVM solves an optimization problem: maximize the margin while minimizing classification error. The margin is `2 / ||w||` for a hyperplane `w^T x + b = 0`. This is a convex optimization problem.

**4) Role of Lagrange Multipliers in SVM**
They convert the constrained primal problem into a dual problem. The dual depends on dot products of samples, enabling the kernel trick.

**5) What are Support Vectors in SVM**
Support vectors are the training samples closest to the decision boundary. They define the margin and fully determine the hyperplane.

**6) What is a Support Vector Classifier (SVC)**
An SVC is the classification variant of SVM. It finds a maximum-margin hyperplane (possibly in a transformed feature space with kernels).

**7) What is a Support Vector Regressor (SVR)**
SVR is the regression variant of SVM. It fits a function within an epsilon-insensitive tube and penalizes points outside it.

**8) What is the Kernel Trick in SVM**
The kernel trick replaces dot products with a kernel function, allowing SVMs to learn nonlinear boundaries without explicitly computing high-dimensional features.

**9) Compare Linear, Polynomial, and RBF Kernels**
- Linear: fast, works well when data is (almost) linearly separable.
- Polynomial: models interactions up to a degree; can be sensitive to degree choice.
- RBF: maps data to infinite-dimensional space; very flexible but sensitive to `gamma`.

**10) Effect of the C parameter in SVM**
`C` controls the penalty for misclassification. High `C` tries to fit training data more strictly (low bias, high variance). Low `C` allows more violations (higher bias, lower variance).

**11) Role of Gamma in RBF Kernel SVM**
`gamma` controls how far the influence of a single training example reaches. High `gamma` creates complex boundaries; low `gamma` gives smoother boundaries.

**12) What is the Naive Bayes classifier, and why is it called "Naive"**
Naive Bayes is a probabilistic classifier based on Bayes' theorem, assuming features are conditionally independent given the class. This independence assumption is "naive" but often works well.

**13) What is Bayes' Theorem**
`P(C|X) = P(X|C) * P(C) / P(X)` where `C` is the class and `X` is the feature vector.

**14) Differences between Gaussian, Multinomial, and Bernoulli Naive Bayes**
- Gaussian: continuous features with normal distribution assumption.
- Multinomial: count data (word counts).
- Bernoulli: binary features (word present/absent).

**15) When should you use Gaussian Naive Bayes**
Use it for continuous features that are roughly normally distributed (e.g., measurements).

**16) Key assumptions of Naive Bayes**
Conditional independence of features given the class, and correct likelihood model (Gaussian/Multinomial/Bernoulli).

**17) Advantages and disadvantages of Naive Bayes**
Advantages: fast, works with small data, robust to irrelevant features. Disadvantages: independence assumption can be violated; probability estimates can be poorly calibrated.

**18) Why is Naive Bayes good for text classification**
Text data is high-dimensional and sparse. Multinomial/Bernoulli NB performs well and is efficient with word counts.

**19) Compare SVM and Naive Bayes for classification**
SVM is discriminative and often achieves higher accuracy with good tuning. Naive Bayes is generative, faster, and works well with limited data or text features.

**20) How does Laplace Smoothing help in Naive Bayes**
It adds a small constant (usually 1) to counts to avoid zero probabilities for unseen features.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import (
    load_iris,
    load_wine,
    load_breast_cancer,
    fetch_california_housing,
    fetch_20newsgroups,
    make_classification,
    make_regression,
)
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    mean_absolute_error,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    log_loss,
    roc_auc_score,
    precision_recall_curve,
    classification_report,
)
from sklearn.svm import SVC, SVR
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.feature_extraction.text import CountVectorizer

RANDOM_STATE = 42


## Theoretical Programs


### 1) Train an SVM Classifier on the Iris dataset and evaluate accuracy


In [ ]:
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=RANDOM_STATE, stratify=iris.target
)
model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale"))
model.fit(X_train, y_train)

pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"Iris SVM accuracy: {acc:.4f}")


### 2) Train two SVM classifiers with Linear and RBF kernels on the Wine dataset, compare accuracies


In [ ]:
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=RANDOM_STATE, stratify=wine.target
)

linear_svm = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0))
rbf_svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale"))

linear_svm.fit(X_train, y_train)
rbf_svm.fit(X_train, y_train)

acc_linear = accuracy_score(y_test, linear_svm.predict(X_test))
acc_rbf = accuracy_score(y_test, rbf_svm.predict(X_test))

print(f"Wine SVM (Linear) accuracy: {acc_linear:.4f}")
print(f"Wine SVM (RBF) accuracy: {acc_rbf:.4f}")


### 3) Train an SVR on a housing dataset and evaluate using MSE


In [ ]:
try:
    housing = fetch_california_housing()
    X, y = housing.data, housing.target
except Exception as e:
    print("California housing download failed, using synthetic regression data.")
    X, y = make_regression(n_samples=2000, n_features=8, noise=10.0, random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

svr = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=10.0, gamma="scale"))
svr.fit(X_train, y_train)

pred = svr.predict(X_test)
mse = mean_squared_error(y_test, pred)
print(f"SVR MSE: {mse:.4f}")


### 4) Train an SVM Classifier with a Polynomial Kernel and visualize the decision boundary


In [ ]:
# Use a 2D synthetic dataset for visualization
X, y = make_classification(
    n_samples=300, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.0, random_state=RANDOM_STATE
)

clf = make_pipeline(StandardScaler(), SVC(kernel="poly", degree=3, C=1.0, gamma="scale"))
clf.fit(X, y)

# Decision boundary plot
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=30)
plt.title("Polynomial Kernel SVM Decision Boundary")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()


### 5) Train a Gaussian Naive Bayes classifier on the Breast Cancer dataset and evaluate accuracy


In [ ]:
bc = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    bc.data, bc.target, test_size=0.2, random_state=RANDOM_STATE, stratify=bc.target
)

nb = GaussianNB()
nb.fit(X_train, y_train)

pred = nb.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"Gaussian NB accuracy: {acc:.4f}")


### 6) Train a Multinomial Naive Bayes classifier for text classification using the 20 Newsgroups dataset


In [ ]:
try:
    newsgroups = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
    vectorizer = CountVectorizer(stop_words="english", max_features=5000)
    X = vectorizer.fit_transform(newsgroups.data)
    y = newsgroups.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    mnb = MultinomialNB()
    mnb.fit(X_train, y_train)
    pred = mnb.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"20 Newsgroups Multinomial NB accuracy: {acc:.4f}")
except Exception as e:
    print("20 Newsgroups download failed. You can run this cell with internet access.")


## Practical Programs


### 1) Train an SVM with different C values and compare decision boundaries visually


In [ ]:
X, y = make_classification(
    n_samples=300, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=0.9, random_state=RANDOM_STATE
)

Cs = [0.1, 1.0, 10.0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, C in zip(axes, Cs):
    clf = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=C, gamma="scale"))
    clf.fit(X, y)
    
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=20)
    ax.set_title(f"C = {C}")

plt.tight_layout()
plt.show()


### 2) Train a Bernoulli Naive Bayes classifier on a binary-feature dataset


In [ ]:
X, y = make_classification(
    n_samples=500, n_features=20, n_informative=10, n_redundant=0,
    n_classes=2, random_state=RANDOM_STATE
)
# Binarize features
X_bin = (X > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_bin, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

bnb = BernoulliNB()
bnb.fit(X_train, y_train)

pred = bnb.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"Bernoulli NB accuracy: {acc:.4f}")


### 3) Feature scaling before SVM and compare with unscaled data


In [ ]:
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

svm_unscaled = SVC(kernel="rbf", C=1.0, gamma="scale")
svm_unscaled.fit(X_train, y_train)
acc_unscaled = accuracy_score(y_test, svm_unscaled.predict(X_test))

svm_scaled = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale"))
svm_scaled.fit(X_train, y_train)
acc_scaled = accuracy_score(y_test, svm_scaled.predict(X_test))

print(f"Unscaled SVM accuracy: {acc_unscaled:.4f}")
print(f"Scaled SVM accuracy: {acc_scaled:.4f}")


### 4) Gaussian Naive Bayes predictions before/after smoothing (variance smoothing)
Note: Laplace smoothing applies to count models; GaussianNB uses `var_smoothing` to stabilize variance.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

nb_default = GaussianNB()
nb_default.fit(X_train, y_train)

nb_smooth = GaussianNB(var_smoothing=1e-6)
nb_smooth.fit(X_train, y_train)

pred_default = nb_default.predict(X_test)
pred_smooth = nb_smooth.predict(X_test)

print("Default accuracy:", accuracy_score(y_test, pred_default))
print("Smoothed accuracy:", accuracy_score(y_test, pred_smooth))


### 5) SVM with GridSearchCV (C, gamma, kernel)


In [ ]:
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

pipeline = make_pipeline(StandardScaler(), SVC())
param_grid = {
    "svc__C": [0.1, 1, 10],
    "svc__gamma": ["scale", 0.1, 0.01],
    "svc__kernel": ["rbf", "linear"],
}

search = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV score:", search.best_score_)
print("Test accuracy:", search.score(X_test, y_test))


### 6) SVM on imbalanced dataset with class weighting


In [ ]:
X, y = make_classification(
    n_samples=1000, n_features=20, n_informative=10, n_redundant=0,
    weights=[0.9, 0.1], random_state=RANDOM_STATE
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

svm_no_weight = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0))
svm_weighted = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, class_weight="balanced"))

svm_no_weight.fit(X_train, y_train)
svm_weighted.fit(X_train, y_train)

pred_no = svm_no_weight.predict(X_test)
pred_wt = svm_weighted.predict(X_test)

print("No weight accuracy:", accuracy_score(y_test, pred_no))
print("Weighted accuracy:", accuracy_score(y_test, pred_wt))
print("Weighted classification report:
", classification_report(y_test, pred_wt))


### 7) Naive Bayes for spam detection using email-like text


In [ ]:
texts = [
    "Win money now", "Claim your free prize", "Limited time offer",
    "Meeting agenda attached", "Project update", "Lunch tomorrow?",
    "Exclusive discount inside", "Invoice for services", "Schedule a call"
]
labels = [1, 1, 1, 0, 0, 0, 1, 0, 0]  # 1 = spam, 0 = ham

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

y = np.array(labels)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_train, y_train)

pred = mnb.predict(X_test)
print("Spam detection accuracy:", accuracy_score(y_test, pred))
print("Predictions:", pred)


### 8) Train an SVM and Naive Bayes on the same dataset and compare accuracy


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0))
nb = GaussianNB()

svm.fit(X_train, y_train)
nb.fit(X_train, y_train)

acc_svm = accuracy_score(y_test, svm.predict(X_test))
acc_nb = accuracy_score(y_test, nb.predict(X_test))

print("SVM accuracy:", acc_svm)
print("Naive Bayes accuracy:", acc_nb)


### 9) Feature selection before Naive Bayes and compare results


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_scaled = MinMaxScaler().fit_transform(X)  # chi2 requires non-negative

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

nb_full = GaussianNB()
nb_full.fit(X_train, y_train)
acc_full = accuracy_score(y_test, nb_full.predict(X_test))

selector = SelectKBest(score_func=chi2, k=10)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

nb_sel = GaussianNB()
nb_sel.fit(X_train_sel, y_train)
acc_sel = accuracy_score(y_test, nb_sel.predict(X_test_sel))

print("Full features accuracy:", acc_full)
print("Selected features accuracy:", acc_sel)


### 10) OvR vs OvO on Wine dataset


In [ ]:
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

base = SVC(kernel="rbf", C=1.0, gamma="scale")
ovr = make_pipeline(StandardScaler(), OneVsRestClassifier(base))
ovo = make_pipeline(StandardScaler(), OneVsOneClassifier(base))

ovr.fit(X_train, y_train)
ovo.fit(X_train, y_train)

acc_ovr = accuracy_score(y_test, ovr.predict(X_test))
acc_ovo = accuracy_score(y_test, ovo.predict(X_test))

print("OvR accuracy:", acc_ovr)
print("OvO accuracy:", acc_ovo)


### 11) Linear, Polynomial, and RBF kernels on Breast Cancer dataset


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

kernels = ["linear", "poly", "rbf"]
for k in kernels:
    svm = make_pipeline(StandardScaler(), SVC(kernel=k, C=1.0, gamma="scale"))
    svm.fit(X_train, y_train)
    acc = accuracy_score(y_test, svm.predict(X_test))
    print(f"Kernel {k} accuracy: {acc:.4f}")


### 12) Stratified K-Fold Cross-Validation for SVM


In [ ]:
X, y = load_wine(return_X_y=True)

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores = cross_val_score(svm, X, y, cv=cv)
print("CV scores:", scores)
print("Average accuracy:", scores.mean())


### 13) Naive Bayes with different prior probabilities


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

nb_default = GaussianNB()
nb_default.fit(X_train, y_train)
acc_default = accuracy_score(y_test, nb_default.predict(X_test))

# Set priors explicitly
nb_prior = GaussianNB(priors=[0.7, 0.3])
nb_prior.fit(X_train, y_train)
acc_prior = accuracy_score(y_test, nb_prior.predict(X_test))

print("Default priors accuracy:", acc_default)
print("Custom priors accuracy:", acc_prior)


### 14) RFE before SVM and compare accuracy


In [ ]:
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# RFE needs a linear estimator
svm_linear = SVC(kernel="linear", C=1.0)
rfe = RFE(estimator=svm_linear, n_features_to_select=5)

X_train_sel = rfe.fit_transform(X_train, y_train)
X_test_sel = rfe.transform(X_test)

svm_full = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0))
svm_sel = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0))

svm_full.fit(X_train, y_train)
svm_sel.fit(X_train_sel, y_train)

acc_full = accuracy_score(y_test, svm_full.predict(X_test))
acc_sel = accuracy_score(y_test, svm_sel.predict(X_test_sel))

print("Full features accuracy:", acc_full)
print("RFE selected accuracy:", acc_sel)


### 15) SVM with Precision, Recall, and F1-Score


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0))
svm.fit(X_train, y_train)

pred = svm.predict(X_test)
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))


### 16) Naive Bayes with Log Loss (Cross-Entropy Loss)


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

nb = GaussianNB()
nb.fit(X_train, y_train)

proba = nb.predict_proba(X_test)
loss = log_loss(y_test, proba)
print("Log Loss:", loss)


### 17) Confusion Matrix for SVM (seaborn)


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0))
svm.fit(X_train, y_train)

pred = svm.predict(X_test)
cm = confusion_matrix(y_test, pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("SVM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


### 18) SVR evaluated with MAE instead of MSE


In [ ]:
try:
    housing = fetch_california_housing()
    X, y = housing.data, housing.target
except Exception:
    X, y = make_regression(n_samples=2000, n_features=8, noise=10.0, random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

svr = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=10.0, gamma="scale"))
svr.fit(X_train, y_train)

pred = svr.predict(X_test)
mae = mean_absolute_error(y_test, pred)
print(f"SVR MAE: {mae:.4f}")


### 19) Naive Bayes with ROC-AUC


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

nb = GaussianNB()
nb.fit(X_train, y_train)

proba = nb.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba)
print("ROC-AUC:", auc)


### 20) Precision-Recall Curve for SVM


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, probability=True))
svm.fit(X_train, y_train)

proba = svm.predict_proba(X_test)[:, 1]
precision, recall, _ = precision_recall_curve(y_test, proba)

plt.figure(figsize=(6, 4))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("SVM Precision-Recall Curve")
plt.grid(True)
plt.show()
